<a href="https://colab.research.google.com/github/masteryongsa82/Portfolio/blob/main/Cipher_Version1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random

# --- Enigma Components Classes ---

class Rotor:
    def __init__(self, wiring, notch_position, ring_setting=0):
        self.wiring = wiring.upper()
        self.notch_position = notch_position.upper()
        self.position = 0  # Initial rotor position (0-25)
        self.ring_setting = ring_setting # Ring setting

        self.inverse_wiring = [''] * 26
        for i, char in enumerate(self.wiring):
            self.inverse_wiring[self._alpha_to_num(char)] = self._num_to_alpha(i)
        self.inverse_wiring = ''.join(self.inverse_wiring)

    def _alpha_to_num(self, char):
        return ord(char) - ord('A')

    def _num_to_alpha(self, num):
        return chr(num + ord('A'))

    def rotate(self):
        self.position = (self.position + 1) % 26

    def turnover(self):
        return self.position == self._alpha_to_num(self.notch_position)

    def encrypt(self, char_in, forward=True):
        offset = self.position - self.ring_setting

        num_in = (self._alpha_to_num(char_in) + offset) % 26

        if forward:
            char_out_mapped = self.wiring[num_in]
        else:
            char_out_mapped = self.inverse_wiring[num_in]

        num_out = (self._alpha_to_num(char_out_mapped) - offset) % 26

        return self._num_to_alpha(num_out)

    def __repr__(self):
        return f"Rotor(position={self.position}, ring_setting={self.ring_setting}, notch='{self.notch_position}', wiring='{self.wiring}')"


class Reflector:
    def __init__(self, wiring):
        self.wiring = wiring.upper()

    def _alpha_to_num(self, char):
        return ord(char) - ord('A')

    def _num_to_alpha(self, num):
        return chr(num + ord('A'))

    def reflect(self, char_in):
        num_in = self._alpha_to_num(char_in)
        char_out = self.wiring[num_in]
        return char_out

    def __repr__(self):
        return f"Reflector(wiring='{self.wiring}')"


class Plugboard:
    def __init__(self, mappings=None):
        self.wiring = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
        self.mappings_str = mappings if mappings else ""

        if mappings:
            pairs = mappings.upper().split()
            for pair in pairs:
                if len(pair) != 2:
                    raise ValueError("Plugboard mappings must be pairs of letters (e.g., 'AG')")
                char1, char2 = pair[0], pair[1]

                if char1 == char2 or self.wiring[ord(char1) - ord('A')] != self._num_to_alpha(ord(char1) - ord('A')) or self.wiring[ord(char2) - ord('A')] != self._num_to_alpha(ord(char2) - ord('A')):
                    raise ValueError(f"Invalid or duplicate plugboard mapping: {pair}")

                idx1 = ord(char1) - ord('A')
                idx2 = ord(char2) - ord('A')

                self.wiring[idx1], self.wiring[idx2] = self.wiring[idx2], self.wiring[idx1]

    def process(self, char):
        idx = ord(char) - ord('A')
        return self.wiring[idx]

    def _num_to_alpha(self, num):
        return chr(num + ord('A'))

    def __repr__(self):
        return f"Plugboard(mappings='{self.mappings_str}')"


class EnigmaMachine:
    def __init__(self, rotor1, rotor2, rotor3, reflector, plugboard=None):
        self.rotors = [rotor1, rotor2, rotor3]
        self.reflector = reflector
        self.plugboard = plugboard if plugboard else Plugboard()

        # Store initial rotor states for reset capability (e.g., for decryption)
        self._initial_rotor_states = [
            {'position': r.position, 'ring_setting': r.ring_setting, 'wiring': r.wiring, 'notch_position': r.notch_position}
            for r in rotors
        ]

    def _rotate_rotors(self):
        if self.rotors[1].turnover(): # Double-stepping mechanism
            self.rotors[0].rotate()
            self.rotors[1].rotate()
        elif self.rotors[2].turnover(): # Standard turnover
            self.rotors[1].rotate()

        self.rotors[2].rotate() # Rightmost rotor always rotates

    def encrypt_character(self, char):
        char = char.upper()
        if not 'A' <= char <= 'Z':
            return char

        self._rotate_rotors()

        char = self.plugboard.process(char)

        for rotor in reversed(self.rotors):
            char = rotor.encrypt(char, forward=True)

        char = self.reflector.reflect(char)

        for rotor in self.rotors:
            char = rotor.encrypt(char, forward=False)

        char = self.plugboard.process(char)

        return char

    def encrypt_message(self, message):
        encrypted_message = ""
        for char in message:
            encrypted_message += self.encrypt_character(char)
        return encrypted_message

    def reset_rotors_to_initial_state(self):
        # Recreate rotors with their initial ring settings and positions
        for i, initial_state in enumerate(self._initial_rotor_states):
            self.rotors[i] = Rotor(
                wiring=initial_state['wiring'],
                notch_position=initial_state['notch_position'],
                ring_setting=initial_state['ring_setting']
            )

    def __repr__(self):
        return f"EnigmaMachine(rotors={self.rotors}, reflector={self.reflector}, plugboard={self.plugboard})"

# --- Enigma Settings Data ---
ROTOR_SETTINGS = {
    'I': {'wiring': 'EKMFLGDQVZNTOWYHXUSPAIBRCJ', 'notch': 'Q'},
    'II': {'wiring': 'AJDKSIRUXBLHWTMCQGZNPYFVOE', 'notch': 'E'},
    'III': {'wiring': 'BDFHJLCPRTXVZNYEIWGAKMUSQO', 'notch': 'V'}
}

REFLECTOR_SETTINGS = {
    'B': 'YRUHQSLDPXNGOKMIEBFZCWVJAT',
    'C': 'FVPJIAOYEDRZWXUGMTQKSNLHBC'
}

# --- Key Setup Logic ---
print("\n--- 에니그마 키 설정 ---")
config_choice = input("키 설정을 수동으로 입력하시겠습니까 (manual) 또는 무작위로 생성하시겠습니까 (random)? [manual/random]: ").lower()

selected_rotors = []
selected_reflector = None
selected_plugboard = None
initial_key_settings_for_reuse = {} # Store parameters for recreating machine

if config_choice == 'manual':
    rotor_order_str = input("사용할 로터 번호를 순서대로 입력하세요 (예: 3 1 2, 공백으로 구분): ").strip().split()
    ring_settings_str = input("각 로터의 링 설정을 입력하세요 (예: 1 2 3, 공백으로 구분): ").strip().split()
    reflector_type = input("사용할 반사판을 입력하세요 (B 또는 C): ").upper()
    plugboard_input_str = input("플러그보드 연결을 입력하세요 (예: AZ BY CX, 공백으로 구분, 연결이 없으면 Enter): ")

    rotor_configs_for_reuse = []
    for i, rotor_num_str in enumerate(rotor_order_str):
        rotor_id = {'1':'I', '2':'II', '3':'III'}.get(rotor_num_str, None)
        if rotor_id is None:
            print(f"경고: 알 수 없는 로터 번호 '{rotor_num_str}'입니다. 기본값으로 로터 I를 사용합니다.")
            rotor_id = 'I'

        wiring = ROTOR_SETTINGS[rotor_id]['wiring']
        notch = ROTOR_SETTINGS[rotor_id]['notch']
        ring_setting = int(ring_settings_str[i]) if i < len(ring_settings_str) else 0

        selected_rotors.append(Rotor(wiring=wiring, notch_position=notch, ring_setting=ring_setting))
        rotor_configs_for_reuse.append({'wiring': wiring, 'notch_position': notch, 'ring_setting': ring_setting})

    reflector_wiring_for_reuse = REFLECTOR_SETTINGS.get(reflector_type, REFLECTOR_SETTINGS['B'])
    selected_reflector = Reflector(wiring=reflector_wiring_for_reuse)

    selected_plugboard = Plugboard(mappings=plugboard_input_str)

    initial_key_settings_for_reuse = {
        'rotors': rotor_configs_for_reuse,
        'reflector_wiring': reflector_wiring_for_reuse,
        'plugboard_mappings': plugboard_input_str
    }

elif config_choice == 'random':
    print("무작위 키 설정을 생성합니다...")

    random_rotor_ids = random.sample(list(ROTOR_SETTINGS.keys()), 3)
    rotor_configs_for_reuse = []
    for rotor_id in random_rotor_ids:
        wiring = ROTOR_SETTINGS[rotor_id]['wiring']
        notch = ROTOR_SETTINGS[rotor_id]['notch']
        ring_setting = random.randint(0, 25) # 0-25 사이의 무작위 링 설정

        selected_rotors.append(Rotor(wiring=wiring, notch_position=notch, ring_setting=ring_setting))
        rotor_configs_for_reuse.append({'wiring': wiring, 'notch_position': notch, 'ring_setting': ring_setting})

    random_reflector_type = random.choice(list(REFLECTOR_SETTINGS.keys()))
    reflector_wiring_for_reuse = REFLECTOR_SETTINGS[random_reflector_type]
    selected_reflector = Reflector(wiring=reflector_wiring_for_reuse)

    all_letters = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
    random.shuffle(all_letters)
    num_plugboard_pairs = random.randint(0, 10)
    plugboard_pairs = []
    for i in range(num_plugboard_pairs):
        if len(all_letters) >= 2:
            char1 = all_letters.pop()
            char2 = all_letters.pop()
            plugboard_pairs.append(f'{char1}{char2}')
    plugboard_input_str = ' '.join(plugboard_pairs)
    selected_plugboard = Plugboard(mappings=plugboard_input_str)
    print(f"무작위 플러그보드 설정: {plugboard_input_str if plugboard_input_str else '없음'}")

    initial_key_settings_for_reuse = {
        'rotors': rotor_configs_for_reuse,
        'reflector_wiring': reflector_wiring_for_reuse,
        'plugboard_mappings': plugboard_input_str
    }

else:
    print("잘못된 선택입니다. 기본 수동 설정으로 진행합니다.")
    # Default to manual if input is invalid (simplified for example)
    rotor_order_str = ['1', '2', '3']
    ring_settings_str = ['0', '0', '0']
    reflector_type = 'B'
    plugboard_input_str = ''

    rotor_configs_for_reuse = []
    for i, rotor_num_str in enumerate(rotor_order_str):
        rotor_id = {'1':'I', '2':'II', '3':'III'}.get(rotor_num_str, 'I')
        wiring = ROTOR_SETTINGS[rotor_id]['wiring']
        notch = ROTOR_SETTINGS[rotor_id]['notch']
        ring_setting = int(ring_settings_str[i])
        selected_rotors.append(Rotor(wiring=wiring, notch_position=notch, ring_setting=ring_setting))
        rotor_configs_for_reuse.append({'wiring': wiring, 'notch_position': notch, 'ring_setting': ring_setting})

    reflector_wiring_for_reuse = REFLECTOR_SETTINGS.get(reflector_type, REFLECTOR_SETTINGS['B'])
    selected_reflector = Reflector(wiring=reflector_wiring_for_reuse)

    selected_plugboard = Plugboard(mappings=plugboard_input_str)

    initial_key_settings_for_reuse = {
        'rotors': rotor_configs_for_reuse,
        'reflector_wiring': reflector_wiring_for_reuse,
        'plugboard_mappings': plugboard_input_str
    }

# 새로운 '키'로 Enigma Machine 인스턴스 생성
# 이 인스턴스는 암호화 작업에 사용됩니다.
rotors = selected_rotors # Rename for clarity for the EnigmaMachine constructor

enigma_active_machine = EnigmaMachine(rotors[0], rotors[1], rotors[2], selected_reflector, selected_plugboard)

print("\n새로운 키 설정으로 Enigma Machine이 초기화되었습니다.")
print(enigma_active_machine)


--- 에니그마 키 설정 ---
키 설정을 수동으로 입력하시겠습니까 (manual) 또는 무작위로 생성하시겠습니까 (random)? [manual/random]: Manual
사용할 로터 번호를 순서대로 입력하세요 (예: 3 1 2, 공백으로 구분): 3 1 2
각 로터의 링 설정을 입력하세요 (예: 1 2 3, 공백으로 구분): 8 7 1
사용할 반사판을 입력하세요 (B 또는 C): b
플러그보드 연결을 입력하세요 (예: AZ BY CX, 공백으로 구분, 연결이 없으면 Enter): az bx cy

새로운 키 설정으로 Enigma Machine이 초기화되었습니다.
EnigmaMachine(rotors=[Rotor(position=0, ring_setting=8, notch='V', wiring='BDFHJLCPRTXVZNYEIWGAKMUSQO'), Rotor(position=0, ring_setting=7, notch='Q', wiring='EKMFLGDQVZNTOWYHXUSPAIBRCJ'), Rotor(position=0, ring_setting=1, notch='E', wiring='AJDKSIRUXBLHWTMCQGZNPYFVOE')], reflector=Reflector(wiring='YRUHQSLDPXNGOKMIEBFZCWVJAT'), plugboard=Plugboard(mappings='az bx cy'))


In [ ]:
# --- 메시지 암호화/복호화 ---

operation_choice = input("\n수행할 작업을 선택하세요 (암호화: encrypt, 복호화: decrypt): ").lower()
user_message = input("\n처리할 메시지를 입력하세요 (대문자 알파벳만): ")

processed_message = ""

# 암호화 또는 복호화 작업을 시작하기 전에 항상 에니그마 기계를 초기 상태로 재설정합니다.
# 이는 암호화와 복호화 모두 동일한 구성된 초기 로터 위치에서 시작하도록 보장합니다.
enigma_active_machine.reset_rotors_to_initial_state()

if operation_choice == 'encrypt':
    print("\n--- 메시지 암호화 ---")
    processed_message = enigma_active_machine.encrypt_message(user_message.upper())
    print(f"원본 메시지: {user_message}")
    print(f"암호화된 메시지: {processed_message}")
elif operation_choice == 'decrypt':
    print("\n--- 메시지 복호화 ---")

    # 에니그마 기계는 상호적입니다. 동일한 키와 시작 로터 위치로 암호문을 암호화하면 복호화됩니다.
    # enigma_active_machine을 이미 초기 상태로 재설정했으므로, 복호화를 위해 직접 사용할 수 있습니다.
    processed_message = enigma_active_machine.encrypt_message(user_message.upper())
    print(f"암호화된 메시지: {user_message}")
    print(f"복호화된 메시지: {processed_message}")
else:
    print("잘못된 작업 선택입니다. 'encrypt' 또는 'decrypt' 중 하나를 입력해주세요.")


수행할 작업을 선택하세요 (암호화: encrypt, 복호화: decrypt): Decrypt

처리할 메시지를 입력하세요 (대문자 알파벳만): shay

--- 메시지 복호화 ---
암호화된 메시지: shay
복호화된 메시지: ABCA
